# Manchester Trading Society
## Order Books


### The simplest type of order book is a:
### 1. Price-Time Priority Algorithm

In [8]:
# start of algorithm

# Two types of orders to order book:
# Limit Orders (LO) and Market Orders (MO)
# Both these orders interact with the Order Book (OB)

# First, create a class for the Order Book 

# OrderBook() class should represent Buy and Sell Limit Orders on the Order Book.
# Which are an ordered sequence of the type DollarsAndShares which is a dataclass 

# DollarsAndShares is a dataclass
# represents pair of dollar amount (dollar: float) and number of shares (shares: int)

# Remember, each security or ticket has their own order book
# Matching engine has multiple order books, which routes the incoming order based on ticket symbol

# DollarsAndShares represents two types of information given a MO (market order) or LO (limit order)
# If LO DollarsAndShares represents a sorted list of Buy and Sell LOs (incoming to order book).
# If MO DollarsAndShares represents pair of total dollars and total shares transcated when a MO 
# is executed on the order book. 

# OrderBook maintains a price-descending sequence of PriceSizePairs for Buy LOs (descending_bids)
# and price-ascending sequence of PriceSizePairs for Sell LOs (ascending_asks)

# We write a basic method to get the OrderBook's highest bid price (method bid_price)
# lowest ask price (method ask_price)
# mid price (method mid_price)
# spread between highest bid price and lowest ask price (method bid_ask_spread)
# and market depth (method market_depth)

# market depth is the number of bids and asks for a specific ticker (equity in this case) 
# at various price points on the order book that are open in the market

# bids = buy, asks = sell

from dataclasses import dataclass, replace
from typing import Sequence, Tuple, Optional, List

@dataclass(frozen=True)
class DollarsAndShares:
    dollars: float
    shares: int

PriceSizePairs = Sequence[DollarsAndShares]

@dataclass(frozen=True)
class OrderBook:

    descending_bids: PriceSizePairs
    ascending_asks: PriceSizePairs

    def bid_price(self) -> float:
        return self.descending_bids[0].dollars

    def ask_price(self) -> float:
        return self.ascending_asks[0].dollars

    def mid_price(self) -> float:
        return (self.bid_price() + self.ask_price()) / 2

    def bid_ask_spread(self) -> float:
        return self.ask_price() - self.bid_price()

    def market_depth(self) -> float:
        return self.ascending_ask[-1].dollars - \
        self.descending_bids[-1].dollars    

    @staticmethod
    def match_orders(
        ps_pairs: PriceSizePairs,
        shares: int
    ) -> Tuple[DollarsAndShares, PriceSizePairs]: # match order function returns a tuple with DollarsAndShares of incoming order and PriceSizePairs on the order book -> I assume the object will be initalised using a ticker value in a later function -> which we build.
        rem_shares: int = shares
        dollars: float = 0.

        for i, d_s in enumerate(ps_pairs):
            this_price: float = d_s.dollars # price for order
            this_shares: int = d_s.shares # number of shares for this order
            dollars += this_price * min(rem_shares, this_shares)

            if rem_shares < this_shares: # when the defined shares removed possible from order book is less than the number of shares submitted in order
                return (
                    DollarsAndShares(dollars=dollars, shares=shares), # defining structure of DollarsAndShares
                    [DollarsAndShares(
                        dollars = this_price,
                        shares = this_shares - rem_shares
                    )] + list(ps_pairs[i+1])
                )
            else:
                rem_shares -= this_shares # if the defined shares in the incoming order meet the shares available in the orderbook, define shares to be removed as number of shared defined in order
        return (
            DollarsAndShares(dollars=dollars, shares=shares - rem_shares), # final number of shares availabe is initial shares in order minus rem_shares defined in logic above
            []      
        )

    # sell limit order function will have to handle the incoming limit orders (shares, price), add it to the order book (add shares and price of this object to orderbook which are PriceSizePairs on the OB)

    def sell_limit_order(self, price: float, shares: int) -> \
            Tuple[DollarsAndShares, OrderBook]:
        index: Optional[int] = next((i for i, d_s
                                     in enumerate(self.descending_bids)
                                     if d_s.dollars < price), None)
        eligible_bids: PriceSizePairs = self.descending_bids \
            if index is None else self.descending_bids[:index]
        ineligible_bids: PriceSizePairs = [] if index is None else self.descending_bids[index:]
        
        d_s, rem_bids = OrderBook.match_order(eligible_bids, shares)
        new_bids: PriceSizePairs = list(rem_bids) + list(ineligible_bids)
        rem_shares: int = shares - d_s.shares

        if rem_shares > 0:
            new_asks: List[DollarsAndShares] = list(self.ascending_asks)
            index1: Optional[int] = next((i for i, d_s
                                          in enumerate(new_asks)
                                          if d_s.dollars >= price), None)
            if index1 is None:
                new_asks.append(DollarsAndShares(
                      dollars=price,
                      shares=rem_shares
                ))
            elif new_asks[index1].dollars != price:
                new_asks.insert(index1, DollarsAndShares(
                     dollars=price,
                     shares=rem_shares
                ))
            else:
                new_asks[index1] = DollarsAndShares(
                      dollars=price,
                      shares=new_asks[index1].shares + rem_shares
                )
            return d_s, OrderBook(
                  ascending_asks=new_asks,
                  descending_bids=new_bids
            )
        else:
            return d_s, replace(
                  self,
                  descending_bids=new_bids
            )
        
    def sell_market_order(
        self,
        shares: int
    ) -> Tuple[DollarsAndShares, OrderBook]:
        d_s, rem_bids = OrderBook.match_order (
            self.descending_bids,
            shares
        )
        return (d_s, replace(self, descending_bids=rem_bids))
    

    def buy_limit_order(self, price: float, shares: int) -> \
            Tuple[DollarsAndShares, OrderBook]:
        index: Optional[int] = next((i for i, d_s 
                                    in enumerate(self.ascending_asks)
                                    if d_s.dollars > price), None)
        eligible_asks: PriceSizePairs = self.ascending_asks \
            if index is None else self.ascending_asks[:index]
        ineligible_asks: PriceSizePairs = [] if index is None else \
            self.ascending_asks[index:]
    
        d_s, rem_asks = OrderBook.match_order(eligible_asks, shares)
        new_asks: PriceSizePairs = list(rem_asks) + list(ineligible_asks)
        rem_shares: int = shares - d_s.shares

        if rem_shares > 0:
            new_bids: List[DollarsAndShares] = list(self.descending_bids)
            index1: Optional[int] = next((i for i, d_s
                                          in enumerate(new_bids)
                                          if d_s.dollars <= price), None)
        
            if index1 is None:
                new_bids.append(DollarsAndShares(
                    dollars=price,
                    shares=rem_shares
                ))

            elif new_bids[index1].dollars != price:
                new_bids.insert(index1, DollarsAndShares(
                    dollars=price,
                    shares=rem_shares
                ))
            else:
                new_bids[index1] = DollarsAndShares(
                    dollars=price,
                    shares=new_bids[index1].shares + rem_shares
                )
            return d_s, replace(
                self,
                ascending_asks = new_asks,
                descending_bids = new_bids
            )
        else:
            return d_s, replace(
                self,
                ascending_asks=new_asks
            )

    # implement buy_market_order

    def buy_market_order(
        self,
        shares: int
    ) -> Tuple[DollarsAndShares, OrderBook]:
        d_s, rem_bids = OrderBook.match_order ( # we remove the ascending asks and shares associated with the tuple when new bids come in and pass them through the matching engine to execute the match
            self.ascending_asks,
            shares
        )
        return (d_s, replace(self, ascending_asks=rem_bids)) # the old ascending asks list is updated -> maybe updated the variable names given this naming convention for actual exchange -> i.e rem_bids could be named updated_asks (as we are actualy return an updated list of asks after matching engine executes, not bids)
                                                             # as this code implementation only includes rem_bids as positive or negative (and does not use rem asks to indicate asks to be removed from the order book when matched, this is why we have to use rem_bids)


    def print_order_book(self) -> None:
        from pprint import pprint

        print()
        print("Bids")
        pprint(self.descending_bids)
        print()
        print("Asks")
        pprint(self.ascending_asks)
        print()


    def display_order_book(self) -> None:
        import matplotlib.pylot as plt

        bid_prices = [d_s.dollars for d_s in self.descending_bids]
        bid_shares = [d_s.shares for d_s in self.descending_bids]
        if self.descending_bids:
            plt.bar(bid_prices, bid_shares, color='blue')

        ask_prices = [d_s.dollar for d_s in self.ascending_asks]
        ask_shares = [d_s.shares for d_s in self.ascending_asks]
        if self.ascending_asks:
            plt.bar(ask_prices, bask_shares, color='red')

        all_prices = sorted(bid_prices + ask_prices)
        all_ticks = ["%d" % x for x in all_prices]
        plt.xticks(all_prices, all_ticks)
        plt.grid(axis='y')
        plt.xlabel("Prices")
        plt.ylabel("Number of Shares")
        plt.title("Order Book")
        plt.show()



    #def match_orders takes input as ps pairs: PriceSizePairs (represents one side of the Order Book)
    # and the number of shares: int to buy/sell. 
    # match_orders return type is a Tuple[DollarsAndShares, PriceSizePairs]
    # defining the number of shares removed from order book and then the dollar amount
        
# Writing a method for LOs and MOs to interact with the OrderBook
# Fundamentally what are LOs? Limits orders only execute on order book when a specific price conditions are met
# MOs execute on the order book when volume of shares are met
# Buy/Sell LOs and Buy/Sell MOs

# pricing is finding the true value of something (an asset or equity in our case)
# then finding out how to best predict this
# one example -> using order book info on bid - asks and spreads to identify optimial trading price
# ml research papers released are not the best models (kinda dog) -> if you are making a ton of cash 


































NameError: name 'OrderBook' is not defined